In [4]:
#Loading in Packages and Data

#Importing Packages
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
import xarray as xr
import os; import time
import pickle
import h5py
###############################################################
def coefs(coefficients,degree):
    coef=coefficients
    coefs=""
    for n in range(degree, -1, -1):
        string=f"({coefficients[len(coef)-(n+1)]:.1e})"
        coefs+=string + f"x^{n}"
        if n != 0:
            coefs+=" + "
    return coefs
###############################################################

# Importing Model Data
check=False
dir='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'

# # dx = 1 km; Np = 1M; Nt = 5 min
# data=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_1km_5min.nc') #***
# parcel=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_pdata_1km_5min_1e6.nc') #***
# res='1km';t_res='5min'
# Np_str='1e6'

# # dx = 1km; Np = 50M
# #Importing Model Data
# dir2='/home/air673/koa_scratch/'
# data=xr.open_dataset(dir2+'cm1out_1km_1min.nc') #***
# parcel=xr.open_dataset(dir2+'cm1out_pdata_1km_1min_50M.nc') #***
# res='1km'; t_res='1min'; Np_str='50e6'

# # dx = 1km; Np = 50M; Nz = 95
# # Importing Model Data
# dir2='/home/air673/koa_scratch/'
# data=xr.open_dataset(dir2+'cm1out_1km_1min_95nz.nc') #***
# parcel=xr.open_dataset(dir2+'cm1out_pdata_1km_1min_95nz.nc') #***
# res='1km'; t_res='1min_95nz'; Np_str='50e6'

# dx = 250km; Np = 50M 
#Importing Model Data
dir2='/home/air673/koa_scratch/'
data=xr.open_dataset(dir2+'cm1out_250m_1min_50M.nc') #***
parcel=xr.open_dataset(dir2+'cm1out_pdata_250m_1min_50M.nc') #***
res='250m'; t_res='1min'; Np_str='50e6'

In [5]:
import sys
dir2='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'
path=dir2+'../Functions/'
sys.path.append(path)

import NumericalFunctions
from NumericalFunctions import * # import NumericalFunctions 
import PlottingFunctions
from PlottingFunctions import * # import PlottingFunctions


# # Get all functions in NumericalFunctions
# import inspect
# functions = [f[0] for f in inspect.getmembers(NumericalFunctions, inspect.isfunction)]
# functions

In [6]:
if res=='1km':
    dir2='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'
if res=='250m':
    dir2='/mnt/lustre/koa/scratch/air673/'
def initiate_array():
    # Define array dimensions (adjust based on your netCDF)
    t_size = len(data['time'])  # Number of timesteps
    z_size = len(data['zh'])    # Number of vertical levels
    y_size = len(data['yh'])    # Number of y-axis points
    x_size = len(data['xh'])    # Number of x-axis points

    out_file = dir2 + 'Variable_Calculation/' + f'Eulerian_VMF_{res}_{t_res}.h5'
    with h5py.File(out_file, 'a') as f:
        for var_name in ['VMF_c', 'VMF_g']:
            if var_name not in f:
                f.create_dataset(var_name, 
                                 (t_size, z_size, y_size, x_size), 
                                 maxshape=(None, z_size, y_size, x_size),
                                 dtype='float64', 
                                 chunks=(1, z_size, y_size, x_size))


            
def add_timestep_at_index(timestep_data_dict, index):
    out_file = dir2 + 'Variable_Calculation/' + f'Eulerian_VMF_{res}_{t_res}.h5'
    with h5py.File(out_file, 'a') as f:
        if 'VMF_c' in timestep_data_dict:
            f['VMF_c'][index] = timestep_data_dict['VMF_c']
        if 'VMF_g' in timestep_data_dict:
            f['VMF_g'][index] = timestep_data_dict['VMF_g']


In [7]:
def GetBinaryArray(t):
    ##############################################
    #READING BACK IN
    out_file2 = dir2 + 'Variable_Calculation/' + f'Eulerian_Binary_Array_{res}_{t_res}.h5'
    
    with h5py.File(out_file2, 'r') as f:
        A_c = f['A_c'][t]  # Loads the entire A_c array into memory
        A_g = f['A_g'][t]  # Loads the entire A_g array into memory
    return A_c,A_g

In [ ]:
####################################
#RUNNING

In [8]:
w_thresh1=0.1
w_thresh2=0.5
qcqi_thresh=1e-6

In [9]:
initiate_array()

#CALCULATING AND APPENDING TO DATA EACH TIMESTEP
for t in range(len(data['time'])):
    if np.mod(t,1)==0: print(f'Current time {t}')

    print('calculating variables')
    w_data=data['winterp'].isel(time=t).data
    rho_data=data['rho'].isel(time=t).data
    [A_c,A_g]=GetBinaryArray(t)
        
    VMF_c=rho_data*w_data*A_c
    VMF_g=rho_data*w_data*A_g

    print('outputting')
    result = {
        'VMF_c': VMF_c,
        'VMF_g': VMF_g
    }
    add_timestep_at_index(result, t)

Current time 0
calculating variables
outputting
Current time 1
calculating variables
outputting
Current time 2
calculating variables
outputting
Current time 3
calculating variables
outputting
Current time 4
calculating variables
outputting
Current time 5
calculating variables
outputting
Current time 6
calculating variables
outputting
Current time 7
calculating variables
outputting
Current time 8
calculating variables
outputting
Current time 9
calculating variables
outputting
Current time 10
calculating variables
outputting
Current time 11
calculating variables
outputting
Current time 12
calculating variables
outputting
Current time 13
calculating variables
outputting
Current time 14
calculating variables
outputting
Current time 15
calculating variables
outputting
Current time 16
calculating variables
outputting
Current time 17
calculating variables
outputting
Current time 18
calculating variables
outputting
Current time 19
calculating variables
outputting
Current time 20
calculating va

In [ ]:
######################################################

In [10]:
# ###############################################
# #READING BACK IN
# out_file = dir2 + 'Variable_Calculation/' + f'Eulerian_VMF_{res}_{t_res}.h5'

# with h5py.File(out_file, 'r') as f:
#     VMF_c = f['VMF_c'][t]  # Loads the entire A_c array into memory
#     VMF_g = f['VMF_g'][t]  # Loads the entire A_g array into memory

In [1]:
# #####################
# #TESTING
# t=100
# z=10
# w_t=data['winterp'].isel(time=t,zh=z).data
# rho_t=data['rho'].isel(time=t,zh=z).data

# out_file = dir2 + 'Variable_Calculation/' + f'Eulerian_Binary_Array_{res}_{t_res}.h5'
# with h5py.File(out_file, 'r') as f:
#     A_c = f['A_c'][t,z]  # Loads the entire A_c array into memory
#     A_g = f['A_g'][t,z]  # Loads the entire A_g array into memory

# out_file = dir2 + 'Variable_Calculation/' + f'Eulerian_VMF_{res}_{t_res}.h5'
# with h5py.File(out_file, 'r') as f:
#     VMF_c = f['VMF_c'][t,z]  # Loads the entire VMF_c array into memory
#     VMF_g = f['VMF_g'][t,z]  # Loads the entire VMF_g array into memory

# print(np.all(rho_t*w_t*A_c==VMF_c))
# print(np.all(rho_t*w_t*A_g==VMF_g))